# 🦷 Session 2 — Simple 3D CBCT Lab

## Real 3D dental data from the course Drive

This short lab uses the **PMCanalSeg_Pilot** dataset already stored in the instructor's Google Drive.

Each available patient folder contains:

- `image.nii.gz` — the 3D CBCT volume
- `label.nii.gz` — the matching 3D segmentation label

### What we will do

**Mount Drive → choose one patient → load CBCT → inspect 3 planes → overlay the segmentation → show a patient-level split**

That is all.

> This is an educational workshop exercise. Do not treat this notebook as a clinically validated segmentation system.

## 1 — Mount Google Drive

Run this cell and approve access to the Google account that contains the course dataset.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — Install and import the small set of packages we need

In [ ]:
!pip -q install nibabel

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib

## 3 — Find the PMCanalSeg course dataset

The expected course location is:

`MyDrive/Dental_Digital_Twin/data/PMCanalSeg_Pilot`

The code first checks that path. If your Drive layout changed, it searches for the folder automatically.

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/Dental_Digital_Twin/data/PMCanalSeg_Pilot")

if not DATA_DIR.exists():
    matches = list(Path("/content/drive/MyDrive").rglob("PMCanalSeg_Pilot"))
    if not matches:
        raise FileNotFoundError(
            "PMCanalSeg_Pilot was not found in this Google Drive. "
            "Use the instructor account or add the dataset folder to My Drive."
        )
    DATA_DIR = matches[0]

patients = sorted([
    p for p in DATA_DIR.glob("Patient_*")
    if (p / "image.nii.gz").exists() and (p / "label.nii.gz").exists()
])

print("Dataset:", DATA_DIR)
print("Patients found:", len(patients))
print([p.name for p in patients])

## 4 — Choose one patient

For the live workshop, start with **Patient_001**.

Change the name later if you want to inspect another case.

In [ ]:
PATIENT = "Patient_001"  # @param ["Patient_001","Patient_002","Patient_003","Patient_004","Patient_005","Patient_006","Patient_007","Patient_008","Patient_009"]

case_dir = DATA_DIR / PATIENT
image_path = case_dir / "image.nii.gz"
label_path = case_dir / "label.nii.gz"

if not image_path.exists() or not label_path.exists():
    raise FileNotFoundError(f"Image/label pair not found for {PATIENT}")

print("Image:", image_path)
print("Label:", label_path)

## 5 — Load the real 3D CBCT and segmentation

A 2D image has height × width.

A CBCT is a **3D volume**:

**X × Y × Z voxels**

The label is another 3D volume aligned with the CBCT.

In [ ]:
img_nii = nib.load(image_path)
lab_nii = nib.load(label_path)

image = img_nii.get_fdata()
label = lab_nii.get_fdata()

print("CBCT shape:", image.shape)
print("Label shape:", label.shape)
print("Voxel spacing (mm):", img_nii.header.get_zooms()[:3])
print("Label values:", np.unique(label))

assert image.shape == label.shape, "Image and label shapes do not match."
print("✓ Image and label have the same dimensions.")

## 6 — View axial, coronal, and sagittal planes

The same patient can be viewed from three orthogonal directions.

This is one reason a CBCT should **not** be treated as hundreds of unrelated 2D images.

In [ ]:
x, y, z = [s // 2 for s in image.shape]

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

ax[0].imshow(image[:, :, z].T, cmap="gray", origin="lower")
ax[0].set_title("Axial")

ax[1].imshow(image[:, y, :].T, cmap="gray", origin="lower")
ax[1].set_title("Coronal")

ax[2].imshow(image[x, :, :].T, cmap="gray", origin="lower")
ax[2].set_title("Sagittal")

for a in ax:
    a.axis("off")

plt.tight_layout()
plt.show()

## 7 — Overlay the 3D annotation

The segmentation label occupies the same 3D space as the CBCT.

For a simple visual check, we overlay all non-zero label voxels on the middle axial slice.

In [ ]:
mask = label > 0
z = image.shape[2] // 2

plt.figure(figsize=(7, 7))
plt.imshow(image[:, :, z].T, cmap="gray", origin="lower")
plt.imshow(mask[:, :, z].T, cmap="autumn", alpha=0.35, origin="lower")
plt.title(f"{PATIENT} — CBCT + segmentation overlay")
plt.axis("off")
plt.show()

## 8 — What does annotation mean in 3D?

For real CBCT annotation in **3D Slicer**:

1. Import the de-identified CBCT.
2. Open **Segment Editor**.
3. Create the target structure.
4. Use tools such as Paint, Threshold, Grow from Seeds, or Fill Between Slices where appropriate.
5. Review the mask in **axial, coronal, sagittal, and 3D views**.
6. Correct errors manually.
7. Perform expert quality control.
8. Export the image and matching label volume.

### Critical rule

**Never transform the image without applying the same spatial transform to its label.**

## 9 — Patient-level splitting

The course Drive currently contains **9 complete patient folders**.

For teaching, we can demonstrate a simple split:

- 6 patients → training
- 1 patient → validation
- 2 patients → test

The exact split below is only a workshop example.

In [ ]:
train_patients = patients[:6]
val_patients   = patients[6:7]
test_patients  = patients[7:9]

print("TRAIN:", [p.name for p in train_patients])
print("VALIDATION:", [p.name for p in val_patients])
print("TEST:", [p.name for p in test_patients])

assert set(train_patients).isdisjoint(test_patients)
print("\n✓ Patients are independent across train and test.")

## Why not split CBCT slices randomly?

### ❌ Wrong

Patient A slices → some in train, some in test.

The model sees nearly the same anatomy in both groups.

### ✅ Correct principle

**Patient A belongs to one split only.**

This rule also applies to repeated scans and derived images from the same patient.

## 10 — Simple 3D AI concept

For a segmentation model, the learning problem is:

**3D CBCT volume → model → 3D segmentation mask**

Typical model families include:

- 3D U-Net
- nnU-Net
- MONAI-based segmentation workflows

For this 2-hour workshop we stop here because the important learning goal is to understand:

**3D data + 3D annotation + geometry + patient-level splitting**

The main Session 2 model-training exercise remains the faster 2D implant-recognition Colab + Landing AI lab.

## ✅ Take-home message

**2D radiograph**
→ pixels
→ label / box / 2D mask

**3D CBCT**
→ voxels
→ 3D mask
→ preserve spacing and geometry
→ split by patient

### Workshop rule

**Do not make the 3D exercise more complicated than the learning objective.**

First understand the data correctly. Model training can come later.